# Using LLM-Prompt-Artifact



This tutorial illustrates the usage of LLM-Prompt-artifact feature.
This feature enables to use LLMs, including specific prompt templates, inside the complete workflow in a simple way.

Whenever an LLM-Prompt artifact is used, there MUST be a definition of:
- What is the prompt template
- Which LLM is used
- What the model’s generation configuration is (if not using the default)

**In this tutorial**
- [Set up the environment](#1.-Set-up-the-environment)
- [Import mlrun library and initialize the project](#2.-Import-mlrun-library-and-initialize-the-project)
- [Configure OpenAI profile](#3.-Configure-OpenAI-profile)
- [Define the LLM model and prompt templates](#4.-Define-the-LLM-model-and-prompt-templates)
- [Create serving graph, shared model, and proxies](#5.-Enable-tracking-deploys-the-function-and-plots-the-graph)
- Enable tracking, deploy the function, and plot the graph
- Deploy the model monitoring application
- Streamlit bot app configuration
- Use the streamlit chatbot to use your LLM model

In [ ]:
!pip install streamlit

## 1. Set up the environment
This section sets up the environment variables required for OpenAI API access, including the base URL and API key

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv("ai_gateway.env")

assert (os.environ.get("OPENAI_API_KEY", None) is not None) and (
        os.environ.get("OPENAI_BASE_URL", None) is not None
    ), (
        "\
    Missing OpenAI credentials, make sure they are set as environment variables."
    )
os.environ["OPENAI_MAX_RETRIES"] = "100"

## 2. Import mlrun library and initialize the project
This initializes the MLRun project

In [ ]:
%config Completer.use_jedi = False

import mlrun
from mlrun import get_or_create_project

image = "mlrun/mlrun"
project_name = "llm-openai-bot"
project = get_or_create_project(project_name, context="./")

This section sets up the necessary datastore profiles for time-series database (TSDB) and stream data,
which are essential for monitoring model performance and detecting drift.
 The `DatastoreProfileV3io` is used for V3IO storages while `DatastoreProfileTDEngine`, `DatastoreProfileKafkaSource` are used in community edition.

In [ ]:
from src.model_monitoring_utils import enable_model_monitoring
enable_model_monitoring(project=project, deploy_histogram_data_drift_app=False, base_period=1)

## 3. Configure OpenAI profile
This section sets up a openAI profile in here we use model gpt-4o-mini, you can change it to the model you want to use.
Make sure to set the environment variables

In [ ]:
from mlrun.datastore.datastore_profile import OpenAIProfile

open_ai_profile = OpenAIProfile(
            name="openai_profile",
            api_key=os.environ.get("OPENAI_API_KEY"),
            organization=os.environ.get("OPENAI_ORG_ID"),
            project=os.environ.get("OPENAI_PROJECT_ID"),
            base_url=os.environ.get("OPENAI_BASE_URL"),
            timeout=os.environ.get("OPENAI_TIMEOUT"),
            max_retries=os.environ.get("OPENAI_MAX_RETRIES"),
        )
project.register_datastore_profile(open_ai_profile)
model_url = f"ds://openai_profile/gpt-4o-mini"

## 4. Define the LLM model and prompt templates
This section defines the LLM model and prompt templates for finance and sports domains. The `finance_prompt_template` and `sport_prompt_template` are structured to guide the LLM in generating responses based on user queries. The templates include a system message that sets the context for the LLM, and a user message that includes the user's ID, tone, depth level, and question.

In [ ]:
from src.llm_prompts import finance_prompt_template, sport_prompt_template

model_artifact = project.log_model(
        "open-ai",
        model_url=model_url,
    )
finance_llm_prompt_artifact = project.log_llm_prompt(
    "finance_llm_prompt",
    prompt_template=finance_prompt_template,
    model_artifact=model_artifact,
    prompt_legend={
        "question": {
            "field": "question",
            "description": "The main financial question or request the user is asking."
        },
        "depth_level": {
            "field": "depth_level",
            "description": "Indicates the level of detail in the answer (e.g., basic, intermediate, advanced)."
        },
        "user_id": {
            "field": "user_id",
            "description": "Unique identifier of the user, useful for personalization and tracking."
        },
        "tone": {
            "field": "tone",
            "description": "The desired style of the response (e.g., formal, friendly, concise, detailed)."
        },
    },
)
sport_llm_prompt_artifact = project.log_llm_prompt(
    "sport_llm_prompt",
    prompt_template=sport_prompt_template,
    model_artifact=model_artifact,
    prompt_legend={
        "question": {
            "field": "question",
            "description": "The main sports or fitness-related question from the user."
        },
        "depth_level": {
            "field": "depth_level",
            "description": "Indicates how in-depth the explanation should be (e.g., beginner, intermediate, expert)."
        },
        "user_id": {
            "field": "user_id",
            "description": "Unique identifier of the user, used for personalization or tracking."
        },
        "tone": {
            "field": "tone",
            "description": "The preferred style or tone of the response (e.g., motivational, professional, casual)."
        },
    },
)

Define the function graph, adding ModelRunnerStep with proxy models for the shared model

In [ ]:
from mlrun.serving import ModelRunnerStep
from mlrun.common.schemas.model_monitoring.constants import ModelEndpointCreationStrategy

function = mlrun.code_to_function(
    name="open-ai-tut",
    kind="serving",
    tag="latest",
    project=project.name,
    filename="./LLM_file.py",
    image=image,
    requirements=["openai==1.77.0"],
)
graph = function.set_topology("flow", engine="async")

model_runner_step = ModelRunnerStep(name="my_model_runner", model_selector="MyModelSelector")

graph.add_shared_model(
    name="shared_llm",
    execution_mechanism="dedicated_process",
    model_class="LLModel",
    model_artifact=model_artifact,
    result_path="outputs",
)

model_runner_step.add_shared_model_proxy(
    endpoint_name="finance_endpoint",
    model_artifact=finance_llm_prompt_artifact,
    shared_model_name="shared_llm",
    model_endpoint_creation_strategy=ModelEndpointCreationStrategy.OVERWRITE
)
model_runner_step.add_shared_model_proxy(
    endpoint_name="sport_endpoint",
    model_artifact=sport_llm_prompt_artifact,
    shared_model_name="shared_llm",
    model_endpoint_creation_strategy=ModelEndpointCreationStrategy.OVERWRITE
)

graph.to(model_runner_step).respond()

## 5. Enable tracking deploys the function and plots the graph
This section enable tracking deploys the function and plots the graph to visualize the flow of the LLM model
Note: We use the deploy_endpoint for the url for the Gradio app

In [ ]:
function.set_tracking(enable_tracking=True)
graph.plot()

In [ ]:
deploy_endpoint = function.deploy()

## 7. Deploy the model monitoring application
This section deploys the model monitoring application, which is responsible for monitoring the performance of the deployed LLM models. The application is set up to monitor the models deployed in the previous step, and it uses the `monitoring_application.py` script to define the monitoring logic. The application is deployed using the `deploy_function` method, which makes it available for monitoring the LLM models in real time.